# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Seminar 6: Validare încrucișată și optimizare cu Optuna</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/06-evaluare-optimizare/seminar-06-validare-incrucisata-optuna.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "06-evaluare-optimizare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Problema de business

Managerul de retenție: *„Fiecare ofertă ne costă bani. Câți clienți să contactăm, cât câștigăm și de unde știu că va funcționa și luna viitoare?”*

- **Rolul vostru:** consultant: alegeți metrica din costurile clientului, evaluați corect și transformați rezultatul în bani.
- **Ce predați la final:** pragul de decizie recomandat și profitul estimat pe date noi, cu ipotezele folosite, într-un rezumat pentru manager.

Demonstrația de mai jos folosește întâi datele medicale `Heart.csv`, unde costul unui caz ratat este evident. Exercițiul final aplică același flux pe problema managerului de retenție (`lab-set1`), cu profitul din curs ca metrică.

## Ce învățăm azi

1. De ce **o singură** împărțire train/test nu este de încredere.
2. Cum funcționează **validarea încrucișată** (cross-validation, CV).
3. Cum alegem **ce optimizăm** (metrica) pornind de la problemă, nu din obișnuință.
4. Cum lăsăm **Optuna** să caute hiperparametrii, cu un buget de timp.
5. Cum evaluăm corect, **o singură dată**, pe datele de test.

Codul de mai jos este un exemplu de referință. La exercițiul final veți cere unui asistent AI să-l adapteze pentru altă problemă (vezi [ghidul AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md)).

## Datele: boala cardiacă (Heart.csv)

Ținta este `AHD` (Yes/No): pacientul are sau nu boală coronariană.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("./date/Heart.csv").iloc[:, 1:].dropna()
df["AHD"] = (df["AHD"] == "Yes").astype(int)
df = pd.get_dummies(df, columns=["ChestPain", "Thal"], drop_first=True)

X, y = df.drop(columns="AHD"), df["AHD"]
print(X.shape, "| proporția bolnavilor:", round(y.mean(), 2))

## 1. O singură împărțire train/test este o loterie

Antrenăm **același model** pe **aceleași date**, schimbând doar felul în care sunt împărțite (`random_state`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
scoruri = []
for seed in range(20):
    Xa, Xt, ya, yt = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    scoruri.append(model.fit(Xa, ya).score(Xt, yt))

print(f"acuratețe între {min(scoruri):.2f} și {max(scoruri):.2f}, în funcție doar de împărțire")

**Concluzie:** un singur scor de test poate fi norocos sau ghinionist. Avem nevoie de o estimare mai stabilă.

## 2. Validarea încrucișată (K-fold)

Împărțim datele de antrenament în *K* părți egale (de obicei 5 sau 10). Antrenăm de *K* ori: de fiecare dată o parte este folosită pentru validare, iar celelalte *K-1* pentru antrenare. Raportăm **media și abaterea standard** a celor *K* scoruri.

```
 tura 1: [VAL][ant][ant][ant][ant]
 tura 2: [ant][VAL][ant][ant][ant]
 tura 3: [ant][ant][VAL][ant][ant]
 tura 4: [ant][ant][ant][VAL][ant]
 tura 5: [ant][ant][ant][ant][VAL]
```

`Stratified` înseamnă că fiecare parte păstrează proporția claselor (important la clase dezechilibrate).

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Regula de aur: setul de test se pune deoparte ACUM și se folosește o singură dată, la final.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=17)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)
scor = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
print("scoruri pe cele 5 ture:", scor.round(2))
print(f"acuratețe CV: {scor.mean():.3f} ± {scor.std():.3f}")

> **De ce `StandardScaler` este în `pipeline`?** Ca scalarea să fie învățată doar din partea de antrenare a fiecărei ture. Dacă am scala tot setul înainte, informație din datele de validare ar „scurge" în antrenare (*data leakage*) și scorul ar fi prea optimist.

## 3. Ce optimizăm? Metrica vine din problemă

Acuratețea tratează toate greșelile la fel. În practică, greșelile au costuri diferite:

| Problemă | Greșeala costisitoare | Ce optimizăm |
|---|---|---|
| diagnostic medical (inimă, diabet, AVC) | să ratăm un bolnav (fals negativ) | **recall** (sau F2) |
| fraudă, spam | alarme false (fals pozitiv) | **precision** |
| churn, clase dezechilibrate | ambele; acuratețea înșală | **F1**, **ROC-AUC** |
| clase echilibrate, costuri egale | - | acuratețe |
| regresie (prețuri) | erori mari / erori relative | RMSE / MAPE |

Pentru Heart.csv, a trimite acasă un pacient bolnav este cea mai gravă greșeală, deci optimizăm **recall**.

**Exercițiu (2 minute):** scrieți într-o propoziție ce metrică ați optimiza pentru proiectul vostru și de ce.

## 4. Optuna: căutarea hiperparametrilor

În loc să încercăm toate combinațiile (grid search, lent), Optuna propune combinații din ce în ce mai bune, învățând din încercările anterioare. Noi definim doar:

- **spațiul de căutare**: ce modele și ce valori ale hiperparametrilor sunt permise;
- **obiectivul**: scorul CV, cu metrica aleasă din problemă;
- **bugetul**: cât timp sau câte încercări.

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

METRICA = "recall"   # aleasă din problemă (vezi tabelul de mai sus)

def construieste_model(trial):
    tip = trial.suggest_categorical("model", ["logreg", "random_forest"])
    if tip == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(
            C=trial.suggest_float("C", 1e-3, 1e2, log=True), max_iter=2000))
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 2, 12),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        random_state=17, n_jobs=-1)

def obiectiv(trial):
    model = construieste_model(trial)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring=METRICA).mean()

studiu = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=17))
studiu.optimize(obiectiv, timeout=60, n_trials=60)   # buget: cel mult un minut

print(f"{len(studiu.trials)} încercări | cel mai bun {METRICA} CV: {studiu.best_value:.3f}")
print("parametri:", studiu.best_params)

In [ ]:
# Cum a progresat căutarea și ce hiperparametri au contat (grafice interactive, merg și în Colab)
optuna.visualization.plot_optimization_history(studiu).show()
optuna.visualization.plot_param_importances(studiu).show()

In [ ]:
# Toate încercările, ca tabel
studiu.trials_dataframe()[["number", "value", "params_model"]].sort_values("value", ascending=False).head(10)

## 5. Evaluarea finală: o singură dată, pe test

Reconstruim cel mai bun model, îl antrenăm pe **tot** setul de antrenament și îl evaluăm pe setul de test pus deoparte la pasul 2.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

cel_mai_bun = construieste_model(optuna.trial.FixedTrial(studiu.best_params))
cel_mai_bun.fit(X_train, y_train)
y_pred = cel_mai_bun.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["sănătos", "bolnav"]))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["sănătos", "bolnav"]);

**Întrebări de discuție**

- Scorul de test este apropiat de scorul CV? Dacă e mult mai mic, ce poate însemna?
- Câți bolnavi au fost ratați (fals negativi)? Este acceptabil pentru un medic?
- Ce s-ar schimba dacă am optimiza `precision` în loc de `recall`?

## 6. Exercițiu cu asistentul AI

Nu scrieți cod de mână. Formulați o cerere (vezi rețeta din [ghidul AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md)) și verificați rezultatul. Exemplu de pornire:

> Folosind setul `lab-set1` (Telco Customer Churn, încărcat cu `incarca("lab-set1")[0]`, ținta este coloana `Churn`), pregătește datele pentru clasificare, apoi folosește Optuna cu validare încrucișată stratificată în 5 ture pentru a compara regresia logistică și random forest. Optimizează **profitul campaniei de retenție**, ca la curs: fiecare client care pleacă și primește oferta aduce +250 lei, fiecare ofertă trimisă degeaba costă −50 lei (folosește un scorer propriu cu `make_scorer`). Buget: 60 de secunde, `random_state=17`. Pune deoparte 20% date de test înainte de optimizare și evaluează pe ele o singură dată, cu raport de clasificare și matrice de confuzie.

**Verificați înainte să acceptați rezultatul:**

- [ ] setul de test a fost separat **înainte** de Optuna și nu a fost folosit în `obiectiv`;
- [ ] transformările (scalare, codificare) sunt în `pipeline` sau învățate doar pe antrenare;
- [ ] metrica optimizată este cea pe care ați argumentat-o;
- [ ] puteți explica diferența dintre scorul CV și scorul de test.